# IEEE 8500-node feeder — OpenDSS timing (GNN2)

**This notebook file:** `IEEE8500_OpenDSS_timing.ipynb` (in your `GNN2` project folder)

**Not** the `8500-node` **folder** — that folder only holds OpenDSS `.dss` data files.

**Goal:** find `Master.dss`, run snapshot power flow, and measure **median `Solve()` time (ms)** to compare later with GNN inference at batch size 1.

| Step | What |
|------|------|
| **1** | Point OpenDSS at `Master.dss` |
| **2** | Load circuit + time 50 snapshot solves |
| **3** | Generate load-type CSVs (`run_loadtype_dataset_8500`) |
| **4** | Build `edge_index` / `edge_attr` (`build_graph_8500`) |
| **5** | Stack node `X` / `Y` tensors (`assemble_dataset_tensors_8500`) |
| **6** | MLP sweep: 5 architectures, best saved (`train_mlp_8500`) |

Run cells **in order** (top to bottom).


## Step 1 — Path to `Master.dss`

The feeder is a **folder of `.dss` files**. The **entry file** is **`Master.dss`**. Your copy lives in **`GNN2/8500-node`**.

**This step:** set `LOCAL_8500_DIR` if you move the folder, run the code cell, and confirm it prints **`Master.dss`**.

**Pass:** `Master.dss` exists at the printed path.


In [1]:
import pathlib
import opendssdirect

# --- edit only if you move the feeder folder ---
LOCAL_8500_DIR = pathlib.Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\8500-node")
LOCAL_8500_MASTER = LOCAL_8500_DIR / "Master.dss"

DSS_8500: pathlib.Path | None = None

if LOCAL_8500_MASTER.is_file():
    DSS_8500 = LOCAL_8500_MASTER
else:
    for base in (
        pathlib.Path(opendssdirect.__file__).parent,
        pathlib.Path(r"C:\Program Files\OpenDSS"),
        pathlib.Path(r"C:\OpenDSS"),
    ):
        if not base.exists():
            continue
        try:
            for f in base.rglob("Master.dss"):
                if "8500" in str(f).lower():
                    DSS_8500 = f
                    break
        except (OSError, PermissionError):
            pass
        if DSS_8500 is not None:
            break

if DSS_8500 is None:
    print("Master.dss not found. Place the full 8500-node folder at:")
    print(" ", LOCAL_8500_DIR)
else:
    print("Using entry file:", DSS_8500.resolve())


Using entry file: C:\Users\alita\OneDrive\Desktop\GNN2\8500-node\Master.dss


**Checkpoint — Step 1 done**

You have the OpenDSS entry file **`Master.dss`**.

**Next:** Step 2 loads the circuit and measures solve time.


## Step 2 — Load circuit and profile `Solve()`

**This step:** `Redirect` `Master.dss`, run one solve (convergence + bus counts), then **50** snapshot solves and report **median / mean / min / max** in **ms**.

**Note:** `from opendssdirect import dss` is required — the **package** is not callable (`import opendssdirect as dss` breaks `dss(...)`).

**Pass:** `Converged: True`, buses ~4876, node-phases ~8500+, and a stable median time.


In [2]:
# Step 2 — load + time snapshot solves
import statistics
import time

import pathlib

from opendssdirect import dss

LOCAL_8500_MASTER = pathlib.Path(r"C:\Users\alita\OneDrive\Desktop\GNN2\8500-node\Master.dss")
_d = globals().get("DSS_8500")
if _d is not None:
    DSS_8500 = _d
elif LOCAL_8500_MASTER.is_file():
    DSS_8500 = LOCAL_8500_MASTER
else:
    DSS_8500 = None
if DSS_8500 is None:
    raise RuntimeError("Run Step 1 or place Master.dss at GNN2/8500-node/Master.dss")

dss(f'redirect "{DSS_8500.resolve()}"')
dss.Solution.Mode(1)
dss.Solution.Solve()

converged = dss.Solution.Converged()
n_buses = len(dss.Circuit.AllBusNames())
all_nodes = dss.Circuit.AllNodeNames()
n_nodes = len([n for n in all_nodes if "." in n and n.split(".")[1] in ("1", "2", "3")])

print("Circuit:", DSS_8500)
print(f"  Converged    : {converged}")
print(f"  Buses        : {n_buses}")
print(f"  Node-phases  : {n_nodes} (phases 1/2/3)")
print(f"  Control iter : {dss.Solution.ControlIterations()}")
print(f"  PF iter      : {dss.Solution.Iterations()}")

times_ms = []
for _ in range(50):
    t0 = time.perf_counter()
    dss.Solution.Solve()
    times_ms.append((time.perf_counter() - t0) * 1000.0)

med = statistics.median(times_ms)
print()
print("Solve() time over 50 runs (ms):")
print(f"  median : {med:.3f} ms")
print(f"  mean   : {statistics.mean(times_ms):.3f} ms")
print(f"  min    : {min(times_ms):.3f} ms")
print(f"  max    : {max(times_ms):.3f} ms")
print()
print(f"→ Rough batch-1 target vs pure Solve(): beat ~{med:.1f} ms/step")


Circuit: C:\Users\alita\OneDrive\Desktop\GNN2\8500-node\Master.dss
  Converged    : True
  Buses        : 4876
  Node-phases  : 8541 (phases 1/2/3)
  Control iter : 1
  PF iter      : 2

Solve() time over 50 runs (ms):
  median : 47.016 ms
  mean   : 49.884 ms
  min    : 42.119 ms
  max    : 80.902 ms

→ Rough batch-1 target vs pure Solve(): beat ~47.0 ms/step


**Checkpoint — Step 2 done**

You now have **median `Solve()` time** for this machine on the 8500-node case. Use it as an order-of-magnitude bar when comparing to a surrogate (plus PyTorch overhead, etc.).

---

## Step 3 — Generate load-type dataset (8500)

Run the **next code cell** to execute `run_loadtype_dataset_8500.py`. It writes the same style of CSVs as `run_loadtype_dataset.py`, but for **`8500-node/Master.dss`**, under **`datasets_gnn2/loadtype_8500/`**.

Edit **`n_samples`** (and seeds) in that cell if you want a smaller test first.

---

## Step 4 — Build static graph tensors (`edge_index`, `edge_attr`)

Run **after Step 3** so `gnn_edges_phase_static.csv` and `gnn_node_index_master.csv` exist. The script **`build_graph_8500.py`** writes **`graph_tensors/edge_index.pt`**, **`edge_attr.pt`**, **`graph_meta.json`**, and **`node_index_map.json`** under **`datasets_gnn2/loadtype_8500/`**.

---

## Step 5 — Stack node features / targets (`X`, `Y`)

Run **after Steps 3–4** so `gnn_node_features_and_targets.csv` exists and **`graph_meta.json`** can validate **N**. **`assemble_dataset_tensors_8500.py`** writes **`dataset_tensors/X.pt`**, **`Y.pt`**, optional **`Y_angle.pt`** (`vang_deg`), and **`tensor_manifest.json`**.

---

## Step 6 — MLP baseline (checklist *Step 5*)

Run **after Step 5** so **`X.pt`** / **`Y.pt`** exist. **`train_mlp_8500.py`** trains **five** MLP shapes on **MSE(|V|) only** (no angle), saves each under **`mlp_sweep_8500/<name>/`**, writes **`sweep_summary.json`**, and copies the **best** test MAE to **`mlp_sweep_8500/best/`**. For a **single** run use `train_mlp_baseline_8500()`. The checklist target is **test MAE under 0.005 pu** on \|V\| — expect to **increase `n_samples`** (e.g. 5k–10k) in Step 3 first.

---

## Later (outside this notebook)

- Train **`train_gnn_8500`** (checklist Step 6), then benchmark inference vs OpenDSS.


In [ ]:
# Step 3 — same pattern as GNN2 notebook: chdir to repo, run dataset script
import os

try:
    import run_loadtype_dataset_8500 as _lt8500

    os.chdir(os.path.dirname(os.path.abspath(_lt8500.__file__)))
except Exception:
    pass

# Same as: exec(open("run_loadtype_dataset_8500.py", encoding="utf-8").read())
from run_loadtype_dataset_8500 import generate_gnn_snapshot_dataset_loadtype_8500

generate_gnn_snapshot_dataset_loadtype_8500(
    n_samples=500,
    master_seed=20260322,
    sigma_load=0.12,
    sigma_pv=0.12,
)

## Step 4 — Static graph from dataset CSVs

**Pass:** `build_graph_8500.build_static_graph_8500()` prints shapes and writes files under `datasets_gnn2/loadtype_8500/graph_tensors/`.

**Next:** Step 5 — stack **`X`** / **`Y`** tensors from `gnn_node_features_and_targets.csv`.

In [ ]:
# Step 4 — edge_index / edge_attr .pt files (requires Step 3 CSVs)
import os

try:
    import build_graph_8500 as _bg
    os.chdir(os.path.dirname(os.path.abspath(_bg.__file__)))
except Exception:
    pass

from build_graph_8500 import build_static_graph_8500

build_static_graph_8500()

## Step 5 — Stacked tensors for training

**Pass:** `assemble_dataset_tensors_8500.assemble_dataset_tensors_8500()` prints shapes and writes **`dataset_tensors/X.pt`**, **`Y.pt`**, **`tensor_manifest.json`**.

Uses the same **14** load-type input columns as `run_gnn3_best7_train.LOADTYPE_FEAT`; target defaults to **`vmag_pu`**, with **`Y_angle.pt`** from **`vang_deg`** when present.

**Next:** Step 6 — five-MLP sweep (`train_mlp_architecture_sweep_8500`).

In [5]:
# Step 5 — X [S,N,F], Y [S,N] aligned with graph node order (needs Steps 3–4)
import os

try:
    import assemble_dataset_tensors_8500 as _ad
    os.chdir(os.path.dirname(os.path.abspath(_ad.__file__)))
except Exception:
    pass

from assemble_dataset_tensors_8500 import assemble_dataset_tensors_8500

assemble_dataset_tensors_8500()

[assemble_dataset_tensors_8500] Saved to C:\Users\alita\OneDrive\Desktop\GNN2\datasets_gnn2\loadtype_8500\dataset_tensors/
  X (500, 8541, 14)  Y (500, 8541)  target='vmag_pu'
  samples=500 (dropped incomplete samples vs N=8541)


{'dataset_dir': 'C:\\Users\\alita\\OneDrive\\Desktop\\GNN2\\datasets_gnn2\\loadtype_8500',
 'num_samples': 500,
 'num_nodes': 8541,
 'num_features': 14,
 'feature_columns': ['electrical_distance_ohm',
  'm1_p_kw',
  'm1_q_kvar',
  'm2_p_kw',
  'm2_q_kvar',
  'm4_p_kw',
  'm4_q_kvar',
  'm5_p_kw',
  'm5_q_kvar',
  'q_cap_kvar',
  'p_pv_kw',
  'q_pv_kvar',
  'p_sys_balance_kw',
  'q_sys_balance_kvar'],
 'target_column': 'vmag_pu',
 'X_path': 'C:\\Users\\alita\\OneDrive\\Desktop\\GNN2\\datasets_gnn2\\loadtype_8500\\dataset_tensors\\X.pt',
 'Y_path': 'C:\\Users\\alita\\OneDrive\\Desktop\\GNN2\\datasets_gnn2\\loadtype_8500\\dataset_tensors\\Y.pt',
 'X_shape': [500, 8541, 14],
 'Y_shape': [500, 8541],
 'sample_ids': [0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  4

## Step 6 — MLP sweep (LaTeX checklist Step 5)

**Pass:** `train_mlp_architecture_sweep_8500()` runs **five** architectures (**|V| MSE only**), saves each under **`mlp_sweep_8500/<name>/`**, **`sweep_summary.json`**, and **`best/`** (lowest test MAE on \|V\|).

Default **100** epochs × 5 runs is slow on CPU; reduce **`epochs`** for a smoke test.

In [ ]:
# Step 6 — five MLP architectures; loss = MSE(|V|) only; best -> mlp_sweep_8500/best/
# After `git pull`, restart kernel OR run this cell once — drops stale cached module (fixes ImportError on Colab).
import importlib
import os
import sys

sys.modules.pop("train_mlp_8500", None)

# Colab: adjust if your clone path differs
if os.path.isdir("/content/GNN-Sandia"):
    os.chdir("/content/GNN-Sandia")
else:
    try:
        import train_mlp_8500 as _mlp
        os.chdir(os.path.dirname(os.path.abspath(_mlp.__file__)))
        sys.modules.pop("train_mlp_8500", None)
    except Exception as e:
        raise RuntimeError(
            "Cannot locate train_mlp_8500.py — open the notebook from the repo root or set os.chdir(...) to GNN-Sandia."
        ) from e

import train_mlp_8500
importlib.reload(train_mlp_8500)
from train_mlp_8500 import train_mlp_architecture_sweep_8500

train_mlp_architecture_sweep_8500(
    epochs=100,
    batch_size=16,
)